# N038 · 归并排序与逆序计数

**目标：** 从有序子问题合并中同时恢复答案和统计量。

**先修：** N037, N013, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 分治；归并；递归树；稳定性；逆序对。

**配套讲解来源：** [同名逐章意见](../../comment/038_merge_sort_and_inversions.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章表面上是学**归并排序**，真正的看点是：**排序的中间过程能顺便统计一个重要指标——逆序对**。逆序对指的是：给定数组，满足 i < j 但 a[i] > a[j] 的位置对 (i, j)，它衡量数组"离有序有多乱"。衍生的姊妹指标是**翻转对**：满足 i < j 但 a[i] > 2*a[j] 的位置对，LeetCode 493 就是它。暴力数逆序对要双重循环 O(n²)，本章靠归并排序的分治结构把统计压到 O(n log n)。

先看具体到数字的例子（“自动测试”部分 的真实数据）：

- 输入：`a = [3,1,2]`，求逆序对数
- 输出：`2`
- 为什么是 2？我们把所有位置对列出来：(0,1) 是 3>1，算一个；(0,2) 是 3>2，算一个；(1,2) 是 1>2，不算。总共 2 个，所以 `count_inversions([3,1,2]) == 2`。

再看翻转对的例子（“运行示例”部分/“自动测试”部分 的真实数据）：

- 输入：`a = [1,3,2,3,1]`，求翻转对数（a[i] > 2*a[j]）
- 输出：`2`
- 为什么是 2？全部 10 个位置对里，只有 (1,4)：3 > 2×1=2，和 (3,4)：3 > 2×1=2 这两对满足；比如 (1,2) 虽然是普通逆序（3>2），但 3 > 2×2=4 不成立，所以不算翻转对——两种指标并不等价，这是本章反复强调的区别。

## 2. 基础知识：先读懂这些词

- **分治（divide and conquer）**：把大问题切成两半，各自解决后再把两份答案合起来。归并排序是"切两半、分别排好、合并"。
- **归并（merge）**：把两个**各自有序**的列表合成一个有序列表，做法是各拿一个指针从头比大小，每次取较小的进输出。
- **递归树（recursion tree）**：递归调用展开成的一棵树：n 个元素分成两半、再分成两半……共约 log₂n 层，每层合并的总工作量是 O(n)，相乘得 O(n log n)。
- **稳定性（stability）**：相等元素保持原有先后。归并时遇到相等取**左半在前**（代码写 `left[i]<=right[j]`），左半元素原本就在右半元素前面，所以归并排序是稳定的。
- **逆序对（inversion）**：i < j 且 a[i] > a[j] 的位置对。有序数组 0 个，倒序数组最多 n(n-1)/2 个。
- **翻转对（reverse pair）**：i < j 且 a[i] > 2*a[j] 的位置对，条件比"大于"更苛刻，不能把逆序对数换算过来，要单独数。
- **单调扫描（双指针）**：在两个有序列表上各放一个指针，指针只前进不后退，总代价线性。本章数跨半对时就靠它。
- **因子 factor**：代码把"普通逆序"和"翻转对"统一成同一个函数，条件写成 `x > factor * y`；factor=1 是普通逆序，factor=2 是翻转对。

## 3. 思路推导：从小例子开始

- **Step 1：切两半。** 取中点 mid = len(a)//2，把数组切成左半 a[:mid] 和右半 a[mid:]。
- **Step 2：递归解决。** 对左右两半分别递归调用同一个函数，返回"排好序的半区 + 这半区内部的计数"。左半内部的逆序对、右半内部的逆序对就这么一层层数出来了。
- **Step 3：数跨半对（本章的精华）。** 排序后的左半 left 和右半 right 都各自有序。我们要数的是"左半某 x、右半某 y 且 x > factor*y"。对每个 x，从左往右扫描 right 的指针 j，只要 x > factor*right[j] 就前进；停在分界处，j 恰好就是"配得上 x 的 y 的个数"，加进总数。关键是 x 处理完后 **j 不清零**：left 有序，下一个 x 只会更大，能与它配对的 y 只会更多，j 接着往右走就行——这就是"单调扫描"。
- **Step 4：合并。** 用标准归并把 left 和 right 合成一个有序列表，随计数结果一起返回给上一层。
- **Step 5：封口。** `count_inversions` 和 `reverse_pairs` 只是给 factor 取 1 或 2 的一行包装。

**手算演示一：`count_inversions([3,1,2])` 全过程。**

1. 切成 left=[3]、right=[1,2]，分别递归。
2. 左半 [3] 只有一个元素，直接返回 ([3], 0)。
3. 右半 [1,2]：切成 [1] 和 [2]，计数扫描 x=1：1 > 1×2=2 不成立，贡献 0；合并得 [1,2]，返回 ([1,2], 0)。
4. 回到顶层：left=[3]、right=[1,2]、已累计 0。跨半计数：x=3，指针 j 前进：3 > 1×1=1 成立，j=1；3 > 1×2=2 成立，j=2；j 到头，贡献 2。合并 [3] 与 [1,2] 得 [1,2,3]。
5. 总数 0 + 0 + 2 = **2**，与第一节手数的结果一致。

**手算演示二：`reverse_pairs([1,3,2,3,1])` 关键两层（factor=2）。** 顶层切成 [1,3] 和 [2,3,1]。右半 [2,3,1] 内部：切成 [2] 和 [3,1]，后者跨半计数 x=3：3 > 2×1=2 成立，贡献 1，右半共 1。顶层跨半：left=[1,3] 有序，right=[1,2,3] 有序，x=1：1 > 2×1 不成立，j 不动，贡献 0；x=3：3 > 2×1=2 成立 j=1，3 > 2×2=4 不成立停，贡献 1。总计 1（右半）+ 1（跨半）= **2**。

**手算演示三（“运行示例”部分 的对照表）：** “运行示例”部分 把 [1,3,2,3,1] 全部 10 个位置对逐一标出两种条件真假，这张表就是两个定义的"人工裁判"：

| i | j | a[i] | a[j] | 普通逆序 a[i]>a[j] | 翻转对 a[i]>2*a[j] |
|---|---|---|---|---|---|
| 0 | 1 | 1 | 3 | 否 | 否 |
| 0 | 2 | 1 | 2 | 否 | 否 |
| 0 | 3 | 1 | 3 | 否 | 否 |
| 0 | 4 | 1 | 1 | 否 | 否 |
| 1 | 2 | 3 | 2 | 是 | 否（3>4 不成立） |
| 1 | 3 | 3 | 3 | 否 | 否 |
| 1 | 4 | 3 | 1 | 是 | 是（3>2 成立） |
| 2 | 3 | 2 | 3 | 否 | 否 |
| 2 | 4 | 2 | 1 | 是 | 否（2>2 不成立） |
| 3 | 4 | 3 | 1 | 是 | 是（3>2 成立） |

普通逆序共 4 个、翻转对共 2 个。注意第 (1,2) 行和第 (2,4) 行：它们是逆序但不是翻转对——"是逆序"推不出"是翻转对"，两种指标必须分开数。

## 4. 核心代码：merge_sort

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def merge_sort(a):
    if len(a) < 2:
        return list(a)
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return out + left[i:] + right[j:]
```

### 逐段读懂代码

### 函数 1：`merge_sort` —— 纯归并排序

```python
def merge_sort(a):
    if len(a) < 2:
        return list(a)
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return out + left[i:] + right[j:]
```

- `if len(a)<2: return list(a)`：递归出口——0 个或 1 个元素的数组天然有序。写 `list(a)` 而不是直接 `return a` 有讲究：它返回一份**副本**，所以 merge_sort 不修改调用方传入的原数组（“自动测试”部分 敢直接 `merge_sort(a)` 对照 `sorted(a)` 而不怕 a 被改，就是这个原因）。
- `mid=len(a)//2; left=merge_sort(a[:mid]); right=merge_sort(a[mid:])`：切两半分别递归排好。切片 `a[:mid]` 会复制，所以整个函数辅助空间 O(n)。
- `while i<len(left) and j<len(right):`：两指针都没走到头时，反复取较小者。
- `if left[i]<=right[j]:`：相等时取左边——左半元素原本就在右半前面，这一步保证了归并排序的**稳定性**。
- `return out+left[i:]+right[j:]`：主循环结束时必有且仅有一边剩了尾巴，把剩余部分整段接上（两边都接也无所谓，空切片无害）。

### 函数 2：`_count_pairs` —— 排序 + 计数的合体递归

（对应程序见下方分段实现与完整代码。）

- `if len(a)<2: return list(a),0`：出口同时返回"排好序的副本"和"计数 0"——不到两个元素就没有位置对。
- `left,count1=...`、`right,count2=...`：递归拿到两半的有序版本和各自内部的计数，先把两个内部计数加起来。
- `for x in left:` 外循环 + `while j<len(right) and x>factor*right[j]: j+=1` 内循环：就是第三节的单调扫描。对每个 x，j 停在第一个不满足条件的 y 上，此时 j 就是与 x 配对的 y 的个数，`count+=j`。j 定义在外层循环之前且从不清零，两个指针各自只前进，总代价 O(len(left)+len(right))。
- 内循环之后的三行是标准归并，与 merge_sort 一字不差；注意归并前先把计数做完——归并会打乱 left/right 的原貌，先数再并。
- `return out+left[i:]+right[j:],count`：把"合并后的有序表"和"本层总计数"一起交还上一层。

### 函数 3、4：一行包装

```python
def count_inversions(a):
    return _count_pairs(a, 1)[1]

def reverse_pairs(a):
    return _count_pairs(a, 2)[1]
```

factor=1 条件是 x > y（普通逆序），factor=2 条件是 x > 2y（翻转对）；`[1]` 表示只取返回元组里的计数、丢掉排好的表。

## 5. 分段实现：按顺序运行

**本章接口：** `merge_sort(a)`、`count_inversions(a)`、`reverse_pairs(a)`

### merge_sort

In [1]:
def merge_sort(a):
    if len(a) < 2:
        return list(a)
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return out + left[i:] + right[j:]

### _count_pairs

In [2]:
def _count_pairs(a, factor):
    if len(a) < 2:
        return (list(a), 0)
    mid = len(a) // 2
    left, count1 = _count_pairs(a[:mid], factor)
    right, count2 = _count_pairs(a[mid:], factor)
    count = count1 + count2
    j = 0
    for x in left:
        while j < len(right) and x > factor * right[j]:
            j += 1
        count += j
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return (out + left[i:] + right[j:], count)

### count_inversions

In [3]:
def count_inversions(a):
    return _count_pairs(a, 1)[1]

### reverse_pairs

In [4]:
def reverse_pairs(a):
    return _count_pairs(a, 2)[1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,3,2,3,1]
show_table(['i','j','a[i]','a[j]','普通逆序','翻转对'],[(i,j,a[i],a[j],a[i]>a[j],a[i]>2*a[j]) for i in range(len(a)) for j in range(i+1,len(a))])

i,j,a[i],a[j],普通逆序,翻转对
0,1,1,3,False,False
0,2,1,2,False,False
0,3,1,3,False,False
0,4,1,1,False,False
1,2,3,2,True,False
1,3,3,3,False,False
1,4,3,1,True,True
2,3,2,3,False,False
2,4,2,1,True,False
3,4,3,1,True,True


## 7. 正确性、适用条件与复杂度

索引对唯一分为左半内部、右半内部、跨两半三类。因factor为正，右半有序使满足x>factor*y的y构成前缀；x递增时此前缀不缩短。

**代价：** 三种主算法 O(n log n) 时间、O(n) 峰值辅助空间及 O(log n) 递归深度。

### 展开理解

**为什么每个逆序对都被数了恰好一次？** 关键在分类。任何一个位置对 (i, j)（i 在前 j 在后）在每一层递归里只有三种可能：i 和 j 都落在左半，都在右半，或者 i 在左半、j 在右半——两者必居其一且只居其一（不可能 i 在右 j 在左，因为切分点把前面的元素全分给左半）。于是：同一半内的对由递归的下一层数；跨半的对由本层的单调扫描数。每一层各数各的、互不重叠又合起来覆盖所有对，总数自然不多不少。递归到 0/1 个元素时没有对，出口计数 0 兜底。

**为什么单调扫描数跨半对是准的？** 两个事实撑住它。事实一：factor 是正数（1 或 2），right 又是有序的，所以对固定的 x，满足 x > factor*right[j] 的那些 right[j] 一定是从头开始连续排的一段（值小的先满足），j 停在段尾，段长恰是配对数。事实二：left 也是有序的，下一个 x 更大，能满足条件的 right[j] 只会更多、不会变少，所以 j 不回退也不会错过——两指针全程单向，又快又不重不漏。

**复杂度是多少？** 每层的总工作量是 O(n)（扫描 + 合并都线性），共约 log₂n 层，时间 O(n log n)。n = 10⁵ 时大约 10⁵ × 17 ≈ 170 万次操作量级，秒内完成；对比暴力双循环的 10¹⁰ 次，等不起。空间上：切片与合并缓冲加起来峰值 O(n)；递归深度是 O(log n)（n=10⁵ 时约 17 层，远够不着 Python 的递归上限）。注意 merge_sort / _count_pairs 都**不改**输入数组（每次都切片复制），这是它和快排（N039，原地）的风格差异。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `assert count_inversions([3,1,2])==2`：**正常值测试（手算锚点）**。(3,1) 和 (3,2) 两对，答案 2，第三节已全程手推。
- `assert reverse_pairs([1,3,2,3,1])==2`：**正常值测试（翻转对手算锚点）**。恰好两对满足 3 > 2×1，第三节表格逐对验证过。
- 穷举块（`for n in range(6): for a in product([-2,-1,0,1,2],repeat=n)`）：**交叉验证测试（暴力当裁判 + 穷举小规模）**。长度 0 到 5、取值 −2..2 的全部数组（共 1+6+36+…+7776 = 9331 个）逐一检查三件事：`merge_sort(a)==sorted(a)`（排序对不对）；`count_inversions(a)==sum(a[i]>a[j] ...)`（与双重循环暴力数的逆序对一致）；`reverse_pairs(a)==sum(a[i]>2*a[j] ...)`（与暴力的翻转对一致）。取值压到 5 种小整数是为了大量制造相等元素和 2 倍关系（如 1 与 2、-1 与 -2、0 与 0），专抓条件边界（`>` 写成 `>=`）这类小数据才暴露的 bug。

In [6]:
assert count_inversions([3, 1, 2]) == 2

assert reverse_pairs([1, 3, 2, 3, 1]) == 2

from itertools import product

for n in range(6):
    for a in product([-2, -1, 0, 1, 2], repeat=n):
        assert merge_sort(a) == sorted(a)
        assert count_inversions(a) == sum((a[i] > a[j] for i in range(n) for j in range(i + 1, n)))
        assert reverse_pairs(a) == sum((a[i] > 2 * a[j] for i in range(n) for j in range(i + 1, n)))

print('N038: 所有本章断言通过')

N038: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 区分普通逆序对与a[i]>2*a[j]。

**练习 2：** 用索引区间减少切片。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（区分普通逆序对与 a[i]>2*a[j]）**：方向提示——找"是逆序但不是翻转对"和"两者都是"的对照例子。最小素材：[3,2] 中 (3,2) 是逆序（3>2）但不是翻转对（3 > 4 不成立）；[3,1] 中 (3,1) 两者都是。用它们说明为什么 factor=1 的答案不能乘某个系数变成 factor=2 的答案：条件阈值随 y 变化，不是简单平移。验证时直接调用本章两个接口，再与手数结果对照。
- **练习 2（用索引区间减少切片）**：方向提示——现在每层递归都 `a[:mid]`、`a[mid:]` 切片复制，空间和常数都吃亏；改成传 `(a, lo, mid, hi, tmp)` 这样的索引区间，递归只传下标、合并时借助一条共享的辅助数组 tmp 倒来倒去。要小心的边界：合并读 lo..mid 与 mid..hi 两段、写回原位；计数逻辑不变。手算示例建议用 [3,1,2] 走一遍新版调用（顶层 lo=0,hi=3,mid=1），对拍老版答案 2。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def merge_sort(a):
    if len(a) < 2:
        return list(a)
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return out + left[i:] + right[j:]

def _count_pairs(a, factor):
    if len(a) < 2:
        return (list(a), 0)
    mid = len(a) // 2
    left, count1 = _count_pairs(a[:mid], factor)
    right, count2 = _count_pairs(a[mid:], factor)
    count = count1 + count2
    j = 0
    for x in left:
        while j < len(right) and x > factor * right[j]:
            j += 1
        count += j
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    return (out + left[i:] + right[j:], count)

def count_inversions(a):
    return _count_pairs(a, 1)[1]

def reverse_pairs(a):
    return _count_pairs(a, 2)[1]

# 示例与回归测试
assert count_inversions([3, 1, 2]) == 2

assert reverse_pairs([1, 3, 2, 3, 1]) == 2

from itertools import product

for n in range(6):
    for a in product([-2, -1, 0, 1, 2], repeat=n):
        assert merge_sort(a) == sorted(a)
        assert count_inversions(a) == sum((a[i] > a[j] for i in range(n) for j in range(i + 1, n)))
        assert reverse_pairs(a) == sum((a[i] > 2 * a[j] for i in range(n) for j in range(i + 1, n)))

print('N038: 所有本章断言通过')

N038: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **912. Sort an Array（排序数组）**：练的是 `merge_sort` 本体——一个稳定、最坏 O(n log n)、不改输入的排序，正好是这题的标准解法之一。
- **493. Reverse Pairs（翻转对）**：就是 `reverse_pairs` 的原题，条件 i<j 且 a[i] > 2*a[j]。这题练的是本章最核心的一招：在归并的框架里用"合并前的单调扫描"数跨半对，把 O(n²) 暴力压成 O(n log n)。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。